# SeraTune TRIBE v2 Worker

**3 cells, no restarts needed.** Run them top to bottom.

**Before you start:**
- Runtime → Change runtime type → **T4 GPU** (or A100 with Colab Pro)
- Have your [HuggingFace token](https://huggingface.co/settings/tokens) ready (accept [LLaMA 3.2-3B license](https://huggingface.co/meta-llama/Llama-3.2-3B) first)
- No tunnel token needed — we use [Cloudflare Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/) (free, no sign-up)

In [ ]:
# ============================================================
# CELL 1: Install everything
# ============================================================
!pip uninstall -y torch torchaudio torchvision numpy neuralset neuraltrain tribev2 click gtts 2>/dev/null

!pip install -q \
  "numpy==2.2.6" \
  "torch==2.6.0" \
  "torchaudio==2.6.0" \
  "torchvision==0.21.0" \
  --index-url https://download.pytorch.org/whl/cu124

!pip install -q \
  "neuralset==0.0.2" \
  "neuraltrain==0.0.2" \
  "x_transformers==1.27.20" \
  "moviepy>=2.2.1" \
  gtts julius Levenshtein transformers huggingface_hub \
  soundfile librosa langdetect spacy

!pip install -q git+https://github.com/facebookresearch/tribev2.git
!pip install -q fastapi uvicorn python-multipart httpx

# Install cloudflared (Cloudflare Tunnel) — no account needed
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

print('\n✅ Packages installed. Run the next cell.')

In [ ]:
# ============================================================
# CELL 2: Start the worker (runs in a fresh subprocess
#          so it gets the new numpy/torch — no restart needed)
# ============================================================

HF_TOKEN = ""       # paste your HuggingFace token here

if not HF_TOKEN:
    HF_TOKEN = input('Enter your HuggingFace token: ')

worker_script = '''
import sys, os, logging, tempfile, time, io, base64, gzip
import numpy as np
import torch
from huggingface_hub import login
from tribev2 import TribeModel
from fastapi import FastAPI, File, HTTPException, UploadFile
import uvicorn

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Auth
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
else:
    print("ERROR: HF_TOKEN not set"); sys.exit(1)

# GPU info
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"
vram = f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "N/A"
print(f"GPU: {gpu} ({vram})")

# Load model
print("Loading TRIBE v2 model...")
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model loaded!")

# FastAPI app
app = FastAPI(title="TRIBE v2 Worker")

@app.get("/health")
async def health():
    return {"status": "ok", "model_loaded": True}

def _compress_preds(preds: np.ndarray) -> str:
    """Gzip + base64 encode a numpy array for efficient transfer."""
    buf = io.BytesIO()
    np.save(buf, preds.astype(np.float32))
    compressed = gzip.compress(buf.getvalue())
    return base64.b64encode(compressed).decode("ascii")

@app.post("/analyze")
async def analyze(audio: UploadFile = File(...)):
    tmp_path = None
    try:
        suffix = os.path.splitext(audio.filename or "audio.wav")[1] or ".wav"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as tmp:
            tmp_path = tmp.name
            content = await audio.read()
            tmp.write(content)
        logger.info("Running inference on %s (%d bytes)...", audio.filename, len(content))
        start = time.time()
        df = model.get_events_dataframe(audio_path=tmp_path)
        preds, segments = model.predict(events=df)
        elapsed = time.time() - start
        compressed = _compress_preds(preds)
        raw_mb = preds.nbytes / 1e6
        comp_mb = len(compressed) / 1e6
        logger.info(
            "Inference complete: %s -> %s in %.1fs (%.1fMB -> %.1fMB compressed)",
            audio.filename, preds.shape, elapsed, raw_mb, comp_mb,
        )
        return {
            "preds_b64gz": compressed,
            "shape": list(preds.shape),
            "n_segments": preds.shape[0],
            "n_vertices": preds.shape[1],
            "inference_time_s": round(elapsed, 2),
        }
    except Exception as e:
        logger.exception("Inference failed")
        raise HTTPException(500, f"Inference failed: {e}")
    finally:
        if tmp_path:
            os.unlink(tmp_path)

uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info")
'''

with open('/tmp/tribe_worker_full.py', 'w') as f:
    f.write(worker_script)

import subprocess, os, re, time, threading

env = os.environ.copy()
env['HF_TOKEN'] = HF_TOKEN

# Start cloudflared tunnel in background to get a public URL
print('Starting Cloudflare tunnel...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8001'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Read tunnel output in a thread to capture the public URL
public_url = None
url_event = threading.Event()

def _read_tunnel_output():
    global public_url
    for line in tunnel_proc.stdout:
        m = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if m and not public_url:
            public_url = m.group(1)
            url_event.set()

tunnel_thread = threading.Thread(target=_read_tunnel_output, daemon=True)
tunnel_thread.start()

# Wait up to 30s for the tunnel URL
url_event.wait(timeout=30)
if not public_url:
    print('ERROR: Could not get tunnel URL from cloudflared.')
    tunnel_proc.terminate()
    raise RuntimeError('Cloudflare tunnel failed to start')

print()
print('=' * 60)
print(f'Cloudflare tunnel ready!')
print(f'')
print(f'  Public URL: {public_url}')
print(f'')
print(f'  Add to your backend .env:')
print(f'    TRIBE_WORKER_URL={public_url}')
print(f'    USE_MOCK_TRIBE=false')
print('=' * 60)
print()

print('Starting TRIBE worker in a fresh process...')
print('(Model loads + server starts — takes a few minutes)\n')

# Run as subprocess — fresh Python process, no stale numpy
proc = subprocess.Popen(
    ['python', '/tmp/tribe_worker_full.py'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Stream output to notebook
try:
    for line in proc.stdout:
        print(line, end='')
except KeyboardInterrupt:
    proc.terminate()
    tunnel_proc.terminate()
    print('\nWorker stopped.')

In [ ]:
# ============================================================
# CELL 3 (optional): If the worker dies, re-run Cell 2.
# This cell is just for notes.
# ============================================================
print('If the worker stopped, just re-run Cell 2.')
print('The model weights are cached in ./cache so reloading is fast.')